Checking the schemas

In [0]:
%run ../00-setup/00_config

In [0]:
USE CATALOG dbr_dev;
USE SCHEMA wikimediademo_bronze;
SHOW TABLES;

In [0]:
USE CATALOG dbr_dev;
USE SCHEMA wikimediademo_silver;
SHOW TABLES;

Checking the tables we have from wiki streaming

In [0]:
-- Data present in the table (Jan table)
DESCRIBE TABLE dbr_dev.wikimediademo_bronze.ref_wikimedia_projects;


In [0]:
SELECT language_code From dbr_dev.wikimediademo_bronze.ref_wikimedia_projects
GROUP BY language_code


In [0]:
-- Checking streaming table (Ivan made table)
DESCRIBE TABLE dbr_dev.wikimediademo_bronze.wikipedia_edits_stream;

In [0]:
DESCRIBE TABLE dbr_dev.wikimediademo_silver.wikipedia_edits

Checking the data from the bronze tables

In [0]:
SELECT * 
FROM dbr_dev.wikimediademo_silver.wikipedia_edits
LIMIT 20;

Bot and Human editors dashboard part:

In [0]:
SELECT 
    CASE 
        WHEN bot = 'true' THEN 'Bot'
        ELSE 'Human'
    END AS editor_type,
    COUNT(*) AS total_edits
FROM dbr_dev.wikimediademo_silver.wikipedia_edits
GROUP BY 1;

Databricks visualization. Run in Databricks to view.

pyspark check if all data is similar to SQL

In [0]:
%python
from pyspark.sql import functions as F
df_query1 = (
    df_silver
    .groupBy(
        F.when(F.col("bot") == True, "Bot")
         .otherwise("Human")
         .alias("editor_type")
    )
    .agg(
        F.count("*").alias("total_edits")
    )
)

display(df_query1)

In [0]:
SELECT 
    COALESCE(r.language_code, s.wiki) AS language,
    COALESCE(r.project_name, 'Wikimedia Project') AS project,
    SUM(CASE WHEN s.bot = true THEN 1 ELSE 0 END) AS bot_edits,
    SUM(CASE WHEN s.bot = false THEN 1 ELSE 0 END) AS human_edits,
    COUNT(*) AS total_edits
FROM dbr_dev.wikimediademo_silver.wikipedia_edits s
LEFT JOIN dbr_dev.wikimediademo_bronze.ref_wikimedia_projects r
    ON s.wiki = r.wiki_code
GROUP BY 1, 2
ORDER BY total_edits DESC
LIMIT 20;

Databricks visualization. Run in Databricks to view.

In [0]:
SELECT 
    user,
    CASE WHEN bot = true THEN 'Bot' ELSE 'Human' END AS editor_type,
    COUNT(*) AS total_edits
FROM dbr_dev.wikimediademo_silver.wikipedia_edits
GROUP BY 1, 2
ORDER BY total_edits DESC
LIMIT 10;

Databricks visualization. Run in Databricks to view.

In [0]:
SELECT 
    CASE WHEN bot = true THEN 'Bot' ELSE 'Human' END AS editor_type,
    ROUND(AVG(ABS(length_new - length_old)), 1) AS avg_changed_bytes,
    MAX(ABS(length_new - length_old)) AS max_changed_bytes
FROM dbr_dev.wikimediademo_silver.wikipedia_edits
GROUP BY 1;

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.